# 8.10 - Agentic RAG

**Module 8 - RAG Systems** | Netsetos GenAI Engineering

This notebook turns passive retrieval into an *agent*: a system that first decides whether to retrieve at all, grades what it gets back, and retries or falls back when the results are weak. You build the pattern eleven ways - a hand-rolled query router, a self-correcting CRAG loop, tool-calling, multi-source routing, and the same ideas expressed in LangGraph and LlamaIndex - then finish with the production concerns (model tiering, caching, a circuit breaker) and an India-specific routing layer.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Install the libraries this lesson leans on: Google's `google-genai` SDK plus the two big agent frameworks (`langgraph`, `llama-index`) and their Gemini adapters. The line is commented out because it is meant for a fresh Colab or virtualenv - uncomment and run it once.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install google-genai numpy langchain langgraph llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai python-dotenv -q  # noqa


**Explanation**

A one-time environment-prep cell, not code that runs a model. Everything downstream imports from these packages, so install them before anything else in a clean environment.

**How the code works - step by step**
- **`# !pip install ...`** - commented `pip` install pulling `google-genai` (the Gemini SDK), `numpy` (vector math for cosine similarity), `langchain`/`langgraph` (the StateGraph agent in cell 7), `llama-index` + its Google adapters (the router engine in cell 8), and `python-dotenv` (loading keys from a `.env` file).
- **`# noqa`** - silences the linter on the shell-magic line.

**In one line:** uncomment and run once on a fresh machine, then leave it commented.

**What you'll see:** (no output - environment prep)

## 1 - Point the SDK at your API key

Every model call in this notebook - routing, grading, generation, embeddings - goes through Gemini, and the SDK reads your key from the environment. This cell wires that up without ever hardcoding a secret.

> **Needs a Gemini API key** (get one at aistudio.google.com and set `GOOGLE_API_KEY`).

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

A tiny configuration cell. `setdefault` only fills in the variable if it is not already set, so a key exported in your shell or loaded from `.env` wins and the empty-string fallback never clobbers it.

**How the code works - step by step**
- **`import os`** - gives access to the process environment.
- **`os.environ.setdefault("GOOGLE_API_KEY", "")`** - registers the variable name the SDK looks for; leaves any real key already in the environment untouched.

**In one line:** make sure `GOOGLE_API_KEY` exists so `genai.Client()` can find it.

**What you'll see:** (no output - environment prep). If the key is blank, later cells that call Gemini will raise an authentication error.

## 2 - A query router that decides whether to retrieve

The first agentic move is *not retrieving when you don't need to*. This router asks the model to bucket each query into DIRECT (answer from general knowledge), DOCS (search internal docs), or WEB (search the live internet) - so a greeting or a math question never burns a retrieval call.

In [ ]:
# QUERY ROUTER - the agent decides: answer directly, search docs, or search the web
# pip install google-genai
from google import genai

client = genai.Client()   # reads GOOGLE_API_KEY from the environment

class QueryRouter:
    def route(self, query):
        prompt = (
            "Classify the query into ONE category. Return ONLY the category name.\n"
            "- DIRECT: general knowledge, math, greetings (no retrieval needed)\n"
            "- DOCS: company-specific info, policies, pricing (search internal docs)\n"
            "- WEB: current events, real-time data (search the web)\n\n"
            f"Query: {query}\nCategory:")
        label = client.models.generate_content(
            model="gemini-2.5-flash", contents=prompt).text.strip().upper().split()[0]
        return label if label in ("DIRECT", "DOCS", "WEB") else "DOCS"   # default: retrieve, don't guess

router = QueryRouter()
tests = [("What is 2+2?", "DIRECT"), ("What is the refund policy?", "DOCS"),
         ("Who won the IPL match yesterday?", "WEB"), ("Hello!", "DIRECT"),
         ("How much is the GenAI course?", "DOCS")]
correct = 0
for q, expected in tests:
    r = router.route(q); correct += (r == expected)
    print(f"  {'ok' if r==expected else 'XX'}  {q[:34]:34} -> {r}")
print(f"\n  routed {correct}/{len(tests)}; DIRECT skips retrieval (saves tokens + avoids noise)")

# Output:
#   ok  What is 2+2?                       -> DIRECT
#   ok  What is the refund policy?         -> DOCS
#   ok  Who won the IPL match yesterday?   -> WEB
#   ok  Hello!                             -> DIRECT
#   ok  How much is the GenAI course?      -> DOCS
#   routed 5/5; DIRECT skips retrieval (saves tokens + avoids noise)

**Explanation**

A single-LLM classifier wrapped in a class. The whole agent is one prompt that returns exactly one label, plus a safety net: anything the model doesn't return cleanly falls back to DOCS, because retrieving is safer than guessing. The bottom of the cell is a small accuracy harness over five labelled queries.

**How the code works - step by step**
- **`genai.Client()`** - one client, reading the key from the environment set in cell 1.
- **`QueryRouter.route`** - builds a classification prompt listing the three categories, calls `gemini-2.5-flash`, and normalizes the reply with `.strip().upper().split()[0]` to grab the first word.
- **the `return ... if label in (...) else "DOCS"` guard** - forces any unexpected output to the DOCS default so the system retrieves rather than hallucinates.
- **`tests` + loop** - runs five `(query, expected)` pairs, marks each `ok`/`XX`, and tallies how many matched.

**In one line:** one prompt, one label, default to retrieval when unsure.

**What you'll see:** Five lines each showing `ok` and the routed label (2+2 -> DIRECT, refund policy -> DOCS, IPL match -> WEB, Hello -> DIRECT, course price -> DOCS), then `routed 5/5; DIRECT skips retrieval`.

## 3 - Self-correcting retrieval (CRAG in miniature)

Retrieval sometimes returns garbage. This cell grades its own results and, if they don't answer the question, rephrases the query and tries again - up to a hard retry cap. It is the smallest honest version of Corrective RAG.

In [ ]:
# SELF-CORRECTING RETRIEVAL - grade the results, rephrase and retry if bad (CRAG in miniature)
# pip install google-genai numpy
import numpy as np
from google import genai

client = genai.Client()
def embed(t): return np.array(client.models.embed_content(model="gemini-embedding-001", contents=t).embeddings[0].values)

class SelfCorrectingRAG:
    def __init__(self, docs, max_retries=2):
        self.docs, self.max_retries = docs, max_retries
        self.embs = [embed(d) for d in docs]

    def _retrieve(self, query, k=2):
        qe = embed(query)
        order = sorted(range(len(self.docs)), key=lambda i: -float(np.dot(qe, self.embs[i])))
        return [self.docs[i] for i in order[:k]]

    def _is_relevant(self, query, docs):
        verdict = client.models.generate_content(model="gemini-2.5-flash",
            contents=f"Do these docs answer the query? YES or NO only.\nQuery: {query}\nDocs: {docs}").text.upper()
        return "YES" in verdict

    def _rephrase(self, query):
        return client.models.generate_content(model="gemini-2.5-flash",
            contents=f"Rephrase this query differently for better search. Return ONLY the query.\n{query}").text.strip()

    def query(self, question):
        q = question
        for attempt in range(self.max_retries + 1):        # hard cap - always terminates
            docs = self._retrieve(q)
            if self._is_relevant(question, docs):
                ans = client.models.generate_content(model="gemini-2.5-flash",
                    contents=f"Answer from context only.\n{docs}\nQ: {question}").text.strip()
                return {"answer": ans, "attempts": attempt + 1, "status": "found"}
            if attempt < self.max_retries:
                q = self._rephrase(question)                # self-correct and retry
        return {"answer": "I could not find relevant information.", "status": "exhausted"}

rag = SelfCorrectingRAG([
    "Refund: full within 7 days, 50% for 8-30 days, none after 30.",
    "GenAI course: 14999 rupees. EMI available via Razorpay.",
    "Prerequisites: basic Python, high-school math."])
for q in ["Can I pay in installments?", "Do you teach blockchain?"]:
    r = rag.query(q)
    print(f"Q: {q}\n  [{r['status']}] attempts={r.get('attempts','-')} -> {r['answer'][:60]}\n")

# Output:
# Q: Can I pay in installments?
#   [found] attempts=2 -> Yes - EMI is available via Razorpay ...
# Q: Do you teach blockchain?
#   [exhausted] attempts=- -> I could not find relevant information.

**Explanation**

A retrieve-grade-retry loop built by hand over an in-memory embedding index. The key design point is the bounded `for` loop: it always terminates after `max_retries + 1` attempts, so a bad query can never spin forever. Grading and rephrasing are each one extra LLM call.

**How the code works - step by step**
- **`embed`** - helper turning any text into a NumPy vector via `gemini-embedding-001`.
- **`__init__`** - stores the docs and pre-embeds them once.
- **`_retrieve`** - embeds the query, ranks docs by dot-product similarity, returns the top `k`.
- **`_is_relevant`** - asks the model YES/NO whether the retrieved docs actually answer the query (the grader).
- **`_rephrase`** - asks the model to reword the query for a better search.
- **`query`** - the loop: retrieve, grade against the *original* question; on YES generate an answer and return with the attempt count; on NO rephrase and retry until the cap, then return an `exhausted` result.

**In one line:** retrieve -> grade -> if bad, rephrase and retry (bounded) -> answer or give up.

**What you'll see:** Two queries: "Can I pay in installments?" returns `[found] attempts=2` with the EMI/Razorpay answer, and "Do you teach blockchain?" returns `[exhausted]` with "I could not find relevant information."

## 4 - RAG as a tool the agent calls itself

Instead of you deciding when to search, hand the model a toolbox and let it choose. Here document search, a calculator, and a clock are registered as callable tools, and Gemini's automatic function calling invokes whichever one (if any) each question needs.

In [ ]:
# RAG AS A TOOL - the agent chooses when to search, compute, or answer directly
# pip install google-genai numpy
import numpy as np
from google import genai
from google.genai import types

client = genai.Client()
docs = ["GenAI course: 14999 rupees, 146 hours, 14 modules.",
        "Refund: full within 7 days, 50% for 8-30 days, none after 30.",
        "Live classes 7 PM IST daily. EMI via Razorpay."]
embs = [np.array(client.models.embed_content(model="gemini-embedding-001", contents=d).embeddings[0].values) for d in docs]

def search_docs(query: str) -> str:
    """Search the Netsetos knowledge base for company policies, pricing, and schedules."""
    qe = np.array(client.models.embed_content(model="gemini-embedding-001", contents=query).embeddings[0].values)
    order = sorted(range(len(docs)), key=lambda i: -float(np.dot(qe, embs[i])))
    return " | ".join(docs[i] for i in order[:2])

def calculate(expression: str) -> str:
    """Evaluate an arithmetic expression, e.g. '14999 * 1.18'."""
    try: return str(eval(expression, {"__builtins__": {}}))   # demo only; use a safe parser in prod
    except Exception: return "Error"

def get_current_time() -> str:
    """Return the current date and time in IST."""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M IST")

# register the Python functions as tools; automatic function calling runs them for you
# (minimal example - add request timeouts, retries, and locked-down tools in production)
chat = client.chats.create(model="gemini-2.5-flash", config=types.GenerateContentConfig(
    tools=[search_docs, calculate, get_current_time],
    system_instruction="You are a Netsetos assistant. Use a tool when it helps. Answer concisely."))

for q in ["What is the refund policy?", "How much is 14999 with 18% GST?", "What time is it?", "Hi there!"]:
    print(f"You: {q}\nBot: {chat.send_message(q).text.strip()[:90]}\n")

# Output:
# You: What is the refund policy?           <- agent calls search_docs
# Bot: Full refund within 7 days, 50% for 8-30 days, none after 30.
# You: How much is 14999 with 18% GST?      <- agent calls calculate
# Bot: 14999 with 18% GST is 17,698.82.
# You: What time is it?                    <- agent calls get_current_time
# Bot: It is 2026-07-04 15:12 IST.
# You: Hi there!                            <- no tool, direct reply
# Bot: Hi! How can I help you today?

**Explanation**

This is the tool-use pattern: plain Python functions become the agent's capabilities, and their docstrings tell the model what each is for. The chat session runs the functions automatically, so retrieval is now just one tool among several rather than a hardwired step.

**How the code works - step by step**
- **`docs` + `embs`** - a tiny three-doc knowledge base, pre-embedded.
- **`search_docs`** - embeds the query and returns the top-2 most similar docs; its docstring advertises it as the company knowledge base.
- **`calculate`** - `eval`s an arithmetic string with builtins stripped (demo-only sandbox).
- **`get_current_time`** - returns the current IST timestamp.
- **`client.chats.create(..., tools=[...])`** - opens a chat with all three functions registered and a system instruction telling the bot to use a tool when it helps.
- **the query loop** - four messages exercise each path: a docs question, a GST calculation, a time question, and a plain greeting.

**In one line:** register Python functions as tools and let the model pick which to call.

**What you'll see:** Four You/Bot exchanges: the refund question triggers `search_docs`, the GST question triggers `calculate` (17,698.82), the time question triggers `get_current_time`, and "Hi there!" gets a direct reply with no tool call.

## 5 - Multi-source RAG: route to the best knowledge base

Real products have several document stores - FAQ, billing, technical. This cell routes each query to the single most relevant source first, then retrieves only within it, so answers stay on-topic and cheaper.

In [ ]:
# MULTI-SOURCE RAG - route each query to the best knowledge base
# pip install google-genai numpy
import numpy as np
from google import genai

client = genai.Client()
def embed(t): return np.array(client.models.embed_content(model="gemini-embedding-001", contents=t).embeddings[0].values)

class MultiSourceRAG:
    def __init__(self): self.sources = {}
    def add_source(self, name, description, docs):
        self.sources[name] = {"desc": description, "docs": docs, "embs": [embed(d) for d in docs]}

    def _route(self, query):
        listing = "\n".join(f"- {n}: {s['desc']}" for n, s in self.sources.items())
        pick = client.models.generate_content(model="gemini-2.5-flash",
            contents=f"Which source best answers the query? Return ONLY its name.\n{listing}\nQuery: {query}").text.strip().lower()
        return next((n for n in self.sources if n.lower() in pick), list(self.sources)[0])

    def query(self, question):
        name = self._route(question); s = self.sources[name]
        qe = embed(question)
        order = sorted(range(len(s["docs"])), key=lambda i: -float(np.dot(qe, s["embs"][i])))
        ctx = "\n".join(s["docs"][i] for i in order[:2])
        ans = client.models.generate_content(model="gemini-2.5-flash",
            contents=f"Answer from context.\n{ctx}\nQ: {question}").text.strip()
        return {"source": name, "answer": ans}

rag = MultiSourceRAG()
rag.add_source("faq", "General FAQs, schedules, onboarding",
    ["Live classes 7 PM IST daily; recordings post within 2 hours.", "Onboarding: activate your account from the welcome email."])
rag.add_source("billing", "Pricing, payments, refunds, EMI",
    ["GenAI: 14999 rupees. EMI via Razorpay.", "Refund: full 7 days, 50% 8-30 days, none after 30."])
rag.add_source("technical", "Prerequisites, curriculum, tools",
    ["Prerequisites: Python basics, high-school math.", "Tools: Colab, Vertex AI, ChromaDB."])

for q in ["Can I get a refund?", "What tools will I learn?", "When are the live classes?"]:
    r = rag.query(q)
    print(f"Q: {q}\n  [{r['source']}] -> {r['answer'][:60]}\n")

# Output:
# Q: Can I get a refund?
#   [billing] -> Full refund within 7 days, 50% for 8-30 days ...
# Q: What tools will I learn?
#   [technical] -> Google Colab, Vertex AI, and ChromaDB.
# Q: When are the live classes?
#   [faq] -> Live classes run at 7 PM IST daily; recordings post within 2 hours.

**Explanation**

A two-stage agent: an LLM router picks the source by name from a description listing, then the usual embed-and-rank retrieval runs inside just that source. It generalizes cell 2's router from three fixed buckets to any set of registered sources.

**How the code works - step by step**
- **`add_source`** - registers a named source with a description and its own pre-embedded docs.
- **`_route`** - builds a bulleted listing of every source's description, asks the model which one fits, and matches the reply back to a real source name (falling back to the first source if nothing matches).
- **`query`** - routes, then embeds the question, ranks that source's docs by dot product, takes the top-2 as context, and generates an answer tagged with the chosen source.
- **the demo** - three sources (faq, billing, technical) and three queries that should each land in a different one.

**In one line:** LLM picks the source by its description, then retrieve inside it.

**What you'll see:** Three queries with their routed source in brackets: refund -> `[billing]`, tools -> `[technical]`, live classes -> `[faq]`, each with a short grounded answer.

## 6 - CRAG: grade, then refine, web-fallback, or both

The full Corrective RAG pattern grades retrieval on a three-way scale - CORRECT, INCORRECT, AMBIGUOUS - and takes a different action for each: trust the docs, replace them with a web search, or blend both.

In [ ]:
# CRAG - Corrective RAG: grade the retrieval, then refine / web-fallback / both
# pip install google-genai numpy
import numpy as np
from google import genai

client = genai.Client()
def embed(t): return np.array(client.models.embed_content(model="gemini-embedding-001", contents=t).embeddings[0].values)

class CRAG:
    def __init__(self, docs): self.docs, self.embs = docs, [embed(d) for d in docs]
    def _retrieve(self, q, k=2):
        qe = embed(q); order = sorted(range(len(self.docs)), key=lambda i: -float(np.dot(qe, self.embs[i])))
        return [self.docs[i] for i in order[:k]]
    def _grade(self, q, docs):                          # the CRAG evaluator - one call
        r = client.models.generate_content(model="gemini-2.5-flash",
            contents=f"Do these docs answer the query? Reply ONE word: CORRECT, INCORRECT, or AMBIGUOUS.\nQ: {q}\nDocs: {docs}").text.strip().upper()
        return r.split()[0] if r.split() else "AMBIGUOUS"
    def _web(self, q): return f"[web result for '{q}']"   # plug in Tavily/Serper in production
    def answer(self, q):
        docs = self._retrieve(q); grade = self._grade(q, docs)
        if grade == "INCORRECT": context = self._web(q)                       # docs bad -> web
        elif grade == "AMBIGUOUS": context = "\n".join(docs) + "\n" + self._web(q)  # use both
        else: context = "\n".join(docs)                                       # docs good
        ans = client.models.generate_content(model="gemini-2.5-flash",
            contents=f"Answer from context.\n{context}\nQ: {q}").text.strip()
        return {"grade": grade, "answer": ans}

crag = CRAG(["Refund: full within 7 days, 50% for 8-30 days, none after 30.",
             "GenAI course: 14999 rupees, EMI via Razorpay."])
for q in ["What is the refund window?", "Who won the cricket match last night?"]:
    r = crag.answer(q)
    print(f"Q: {q}\n  grade={r['grade']} -> {r['answer'][:60]}\n")

# Output:
# Q: What is the refund window?
#   grade=CORRECT -> Full refund within 7 days, 50% for 8-30 days ...
# Q: Who won the cricket match last night?
#   grade=INCORRECT -> That is not in the local docs; the web-search fallback supplies it.

**Explanation**

A single grader call drives a branch. Unlike cell 3's binary retry loop, CRAG's three verdicts map to three context-assembly strategies, which is the actual CRAG algorithm. The web search is stubbed so the cell runs offline; in production you drop in Tavily or Serper.

**How the code works - step by step**
- **`_retrieve`** - standard embed-and-rank top-`k`.
- **`_grade`** - one LLM call returning CORRECT / INCORRECT / AMBIGUOUS (the CRAG evaluator).
- **`_web`** - placeholder returning a fake web result string.
- **`answer`** - the branch: INCORRECT swaps in web results, AMBIGUOUS concatenates docs *and* web, CORRECT keeps the docs; then it generates from whichever context was assembled and returns the grade alongside the answer.

**In one line:** grade the retrieval three ways, then pick docs, web, or both before answering.

**What you'll see:** Two queries: the refund-window question grades `CORRECT` and answers from local docs; the cricket-match question grades `INCORRECT` and routes to the web-search fallback.

## 7 - The same CRAG loop as a LangGraph StateGraph

Everything you built by hand in cell 3 is really a state machine, and LangGraph is the standard way to express it. This cell wires retrieve -> grade -> (rewrite -> retrieve) loop -> generate as explicit nodes and edges, with a conditional edge as the loop.

In [ ]:
# LANGGRAPH AGENTIC RAG - the CRAG self-correcting loop as a StateGraph
# pip install langgraph langchain-google-genai
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model

llm = init_chat_model("google_genai:gemini-2.5-flash")

# tiny in-memory retriever so this cell is self-contained;
# swap in your real vector store (Lesson 8.1)
_DOCS = [
    "Refund: full within 7 days, 50% for 8-30 days, none after 30.",
    "GenAI course: 14999 rupees. EMI available via Razorpay.",
]
class _Retriever:
    def invoke(self, query):
        return _DOCS
retriever = _Retriever()

class State(TypedDict):
    question: str
    original_question: str
    documents: List[str]
    generation: str
    retries: int

def retrieve(state):
    return {"documents": retriever.invoke(state["question"])}       # your vector store (Lesson 8.1)

def grade(state):                                              # keep docs only if relevant
    verdict = llm.invoke(f"Relevant to '{state['question']}'? YES/NO.\n{state['documents']}").content.upper()
    return {"documents": state["documents"] if "YES" in verdict else []}

def rewrite(state):
    better = llm.invoke(f"Rewrite for better retrieval: {state['question']}").content
    return {"question": better, "retries": state.get("retries", 0) + 1}

def generate(state):
    q = state.get("original_question", state["question"])   # answer the ORIGINAL question, not the rewrite
    if not state["documents"]:                          # exhausted with no relevant docs
        return {"generation": "I could not find relevant information."}
    return {"generation": llm.invoke(f"Answer from context.\n{state['documents']}\nQ: {q}").content}

def decide(state):                                             # the conditional edge = the loop
    if state["documents"]: return "generate"
    return "rewrite" if state.get("retries", 0) < 2 else "generate"   # hard cap

g = StateGraph(State)
for name, fn in [("retrieve", retrieve), ("grade", grade), ("rewrite", rewrite), ("generate", generate)]:
    g.add_node(name, fn)
g.add_edge(START, "retrieve"); g.add_edge("retrieve", "grade")
g.add_conditional_edges("grade", decide, {"generate": "generate", "rewrite": "rewrite"})
g.add_edge("rewrite", "retrieve"); g.add_edge("generate", END)
app = g.compile()   # add checkpointer=PostgresSaver(...) for durable state + HITL

result = app.invoke({"question": "What is the refund policy?",
                     "original_question": "What is the refund policy?", "retries": 0})
print(result["generation"])

# Output:
# Full refund within 7 days, 50% for 8-30 days, none after 30.

**Explanation**

A declarative graph rather than an imperative loop. A typed `State` dict flows between node functions; each node returns only the state keys it changes. The conditional edge (`decide`) is where the retry loop and its hard cap live, making the control flow inspectable and, in production, checkpointable.

**How the code works - step by step**
- **`_Retriever`** - a stub returning two fixed docs so the cell is self-contained; swap in your real Lesson-8.1 vector store.
- **`State`** - a `TypedDict` carrying the question, the original question, documents, generation, and a retry counter.
- **`retrieve`** - fetches docs for the current question.
- **`grade`** - asks YES/NO if the docs are relevant; empties `documents` on NO.
- **`rewrite`** - reformulates the question and increments `retries`.
- **`generate`** - answers the *original* question from docs, or returns the not-found message when docs are empty.
- **`decide`** - the conditional edge: go to `generate` if docs survived, else `rewrite` until 2 retries then `generate` (the hard cap).
- **graph wiring + `compile()`** - adds the nodes, the START/END edges, the conditional edge, and the `rewrite -> retrieve` back-edge that forms the loop.

**In one line:** the retrieve/grade/rewrite/generate loop as an explicit, checkpointable graph.

**What you'll see:** One line: the refund answer - "Full refund within 7 days, 50% for 8-30 days, none after 30."

## 8 - The same routing in a few lines of LlamaIndex

Where cell 5 routed by hand, LlamaIndex ships the pattern as `RouterQueryEngine`: register each index as a described tool and an LLM selector picks one per query. This is the framework-level version of multi-source RAG.

In [ ]:
# LLAMAINDEX AGENTIC RAG - route each query to the right index in a few lines
# pip install llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai
from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.tools import QueryEngineTool
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

Settings.llm = GoogleGenAI(model="gemini-2.5-flash")
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

from llama_index.core import Document
billing_index = VectorStoreIndex.from_documents([
    Document(text="GenAI course: 14999 rupees. EMI available via Razorpay."),
    Document(text="Refund: full within 7 days, 50% for 8-30 days, none after 30."),
])
tech_index = VectorStoreIndex.from_documents([
    Document(text="Prerequisites: Python basics, high-school math."),
    Document(text="Tools: Colab, Vertex AI, ChromaDB."),
])
tools = [
    QueryEngineTool.from_defaults(query_engine=billing_index.as_query_engine(),
        description="Pricing, refunds, EMI, and payment questions."),
    QueryEngineTool.from_defaults(query_engine=tech_index.as_query_engine(),
        description="Prerequisites, curriculum, and tools."),
]
router = RouterQueryEngine(selector=LLMSingleSelector.from_defaults(), query_engine_tools=tools)

print(router.query("Can I pay in installments?"))     # -> routed to the billing tool
# For "compare X vs Y" queries, use SubQuestionQueryEngine to decompose + merge.
# For a full tool-using agent, use FunctionAgent / AgentWorkflow (the old Agent classes are retired).

# Output:
# EMI is available via Razorpay; installment plans are supported.

**Explanation**

A declarative router built from LlamaIndex primitives. You set global LLM/embedding defaults, build two small vector indexes, wrap each as a described `QueryEngineTool`, and let `LLMSingleSelector` route. It shows how little code the pattern takes once a framework owns the plumbing.

**How the code works - step by step**
- **`Settings.llm` / `Settings.embed_model`** - set Gemini as the global model and embedder for every index.
- **`billing_index` / `tech_index`** - two `VectorStoreIndex`es built from a couple of `Document`s each.
- **`QueryEngineTool.from_defaults(...)`** - wraps each index's query engine with a natural-language `description` the selector reads.
- **`RouterQueryEngine` + `LLMSingleSelector`** - the selector reads the descriptions and routes each query to exactly one tool.
- **`router.query(...)`** - sends a payment question, which should route to the billing tool.

**In one line:** describe each index as a tool and let the selector route to one.

**What you'll see:** One answer for "Can I pay in installments?" routed to the billing tool: "EMI is available via Razorpay; installment plans are supported."

## 9 - Multi-agent RAG with a supervisor (use sparingly)

Sometimes one agent isn't enough and you want specialists - a billing agent, a tech agent, a web agent - each owning its own sources. A supervisor routes each query to exactly one. The cell also flags when this heavier pattern is actually worth it.

In [ ]:
# MULTI-AGENT RAG - a supervisor routes to specialist agents (use sparingly)
# pip install langchain-google-genai
from langchain.chat_models import init_chat_model

llm = init_chat_model("google_genai:gemini-2.5-flash")

def supervisor(query):
    # the supervisor picks ONE specialist; each is a full RAG agent over its own sources
    return llm.invoke(
        "Route to ONE agent: billing_agent, tech_agent, or web_agent. Reply the name only.\n"
        f"Query: {query}").content.strip()

for q in ["Refund policy?", "What tools will I learn?", "Latest Python version?"]:
    print(f"  {q:26} -> {supervisor(q)}")

# Reach for multi-agent ONLY when a single agent + tools cannot cope:
# insufficient context window, per-agent models, or heterogeneous parallel retrieval.

# Output:
#   Refund policy?             -> billing_agent
#   What tools will I learn?   -> tech_agent
#   Latest Python version?     -> web_agent

**Explanation**

A minimal supervisor: one LLM call that returns which specialist should handle the query. It is deliberately thin to make the point that multi-agent is just routing plus per-agent scope - and the comments warn you to reach for it only when a single agent with tools genuinely cannot cope.

**How the code works - step by step**
- **`init_chat_model("google_genai:gemini-2.5-flash")`** - the LangChain model handle for the supervisor.
- **`supervisor`** - one prompt asking the model to reply with just one of `billing_agent`, `tech_agent`, or `web_agent`.
- **the loop** - three queries, each printed next to its chosen specialist.
- **the trailing comment** - lists the only real justifications for multi-agent: context-window limits, per-agent models, or heterogeneous parallel retrieval.

**In one line:** a supervisor LLM names one specialist per query - reach for it only when a single agent can't cope.

**What you'll see:** Three routed lines: refund -> `billing_agent`, tools -> `tech_agent`, latest Python version -> `web_agent`.

## 10 - Production concerns: tiering, caching, and a circuit breaker

Agentic RAG makes many cheap classification calls and a few expensive generation calls, can repeat identical queries, and can loop. This cell shows the three guardrails that address each: model tiering, a semantic cache, and a circuit breaker.

In [ ]:
# PRODUCTION AGENTIC RAG - model tiering, semantic caching, and a circuit breaker
import hashlib

def route_model(task):
    # cheap tier for the many classification calls; frontier tier only for generation
    return "gemini-2.5-flash-lite" if task in ("grade", "route") else "gemini-2.5-flash"

class SemanticCache:
    def __init__(self): self.store = {}
    def get_or_run(self, query, fn):
        key = hashlib.sha256(query.lower().encode()).hexdigest()   # demo: exact-match hash
        # a real semantic cache embeds the query and returns a cached answer above a cosine-similarity threshold
        if key in self.store: return self.store[key], True       # cache hit - no agent run
        self.store[key] = fn(query); return self.store[key], False

class CircuitBreaker:
    """CLOSED -> DEGRADED -> OPEN as loops climb. DEGRADED is where you ALSO add hallucination/cost checks (stubbed here)."""
    def __init__(self, max_loops=3): self.max_loops, self.loops, self.state = max_loops, 0, "CLOSED"
    def step(self):
        self.loops += 1
        if self.loops >= self.max_loops: self.state = "OPEN"            # stop - fall back
        elif self.loops >= self.max_loops - 1: self.state = "DEGRADED"
        return self.state

print("grade ->", route_model("grade"), "| generate ->", route_model("generate"))
cache = SemanticCache(); hits = 0
for q in ["refund?", "refund?", "price?"]:
    _, hit = cache.get_or_run(q, lambda s: f"answer({s})"); hits += int(hit)
print(f"cache hits: {hits}/3")
cb = CircuitBreaker(max_loops=3)
print("loop states:", [cb.step() for _ in range(3)])

# Output:
# grade -> gemini-2.5-flash-lite | generate -> gemini-2.5-flash
# cache hits: 1/3
# loop states: ['CLOSED', 'DEGRADED', 'OPEN']

**Explanation**

A trio of small, dependency-free building blocks - no model calls here, just the control-plane scaffolding you'd wrap around the earlier agents. Each is deliberately stubbed (exact-match hash instead of true semantic cache; state machine instead of real health checks) so you can see the shape before wiring in the real thing.

**How the code works - step by step**
- **`route_model`** - sends grading/routing tasks to the cheap `gemini-2.5-flash-lite` tier and generation to `gemini-2.5-flash`.
- **`SemanticCache.get_or_run`** - hashes the lowercased query; on a hit returns the stored answer with no agent run, otherwise runs the function and caches it (the comment notes a real version embeds and compares by cosine threshold).
- **`CircuitBreaker.step`** - counts loops and walks CLOSED -> DEGRADED -> OPEN, where OPEN means stop and fall back; DEGRADED is where you'd add hallucination/cost checks.
- **the demo** - prints the tier routing, runs three queries (two identical) through the cache, and steps the breaker three times.

**In one line:** cheap tier for the many small calls, cache repeats, and trip a breaker before loops run away.

**What you'll see:** `grade -> gemini-2.5-flash-lite | generate -> gemini-2.5-flash`, then `cache hits: 1/3` (the repeated "refund?"), then `loop states: ['CLOSED', 'DEGRADED', 'OPEN']`.

## 11 - India-specific routing: language + regulatory domain

For an Indian audience, two routing signals matter before retrieval: which language the query is in (to pick generation and embedding models) and which regulatory domain it touches (GST, income tax, SEBI). This cell does both with pure string logic - no LLM call needed.

In [ ]:
# INDIA AGENTIC RAG - language + regulatory-domain routing (cost-optimized)
DOMAINS = {
    "gst": ["gst", "input tax credit", "gstr"],
    "income_tax": ["income tax", "tds", "assessment year"],
    "sebi": ["sebi", "mutual fund", "listing"],
}

def detect_lang(text):
    return "hi" if any("ऀ" <= c <= "ॿ" for c in text) else "en"   # Devanagari range

def route_domain(query):
    q = query.lower()
    for domain, kws in DOMAINS.items():
        if any(k in q for k in kws): return domain
    return "general"

for q in ["GST input tax credit rules?", "रिफंड कैसे मिलेगा?", "SEBI mutual fund norms?"]:
    print(f"  lang={detect_lang(q)}  domain={route_domain(q):11}  | {q}")

# Indic stack: Sarvam open-weight models for generation (verify the current lineup),
# Vyakyarth embeddings for Hindi retrieval, sovereign cloud (Krutrim/Yotta) for residency.

# Output:
#   lang=en  domain=gst          | GST input tax credit rules?
#   lang=hi  domain=general      | रिफंड कैसे मिलेगा?
#   lang=en  domain=sebi         | SEBI mutual fund norms?

**Explanation**

A deterministic, zero-cost pre-router. Language detection is a Unicode-range check for Devanagari; domain routing is keyword matching against a small dictionary. It's a reminder that the cheapest routing is often plain code, and it points at the Indic model stack you'd plug in downstream.

**How the code works - step by step**
- **`DOMAINS`** - a dict mapping each regulatory domain to its trigger keywords.
- **`detect_lang`** - returns `hi` if any character falls in the Devanagari Unicode block, else `en`.
- **`route_domain`** - lowercases the query and returns the first domain whose keywords appear, else `general`.
- **the loop** - three queries (English GST, Hindi refund, English SEBI) printed with detected language and domain.
- **the trailing comment** - names the Indic production stack: Sarvam models, Vyakyarth Hindi embeddings, sovereign cloud for data residency.

**In one line:** cheap deterministic language + domain routing before any model call.

**What you'll see:** Three aligned lines: `lang=en domain=gst` (GST query), `lang=hi domain=general` (the Hindi refund query), and `lang=en domain=sebi` (SEBI query).

You have now seen agentic RAG from every angle: the core moves are *route before you retrieve*, *grade what you retrieved*, and *retry or fall back with a hard cap so the loop always ends*. Cells 2-6 build those moves by hand with the raw Gemini SDK; cells 7-8 show that LangGraph and LlamaIndex are just packaging for the same graph; cells 9-11 add the production layer - supervisors, model tiering, semantic caching, circuit breakers, and language/domain routing. Next in Module 8 you will wire these agents onto the real vector store from Lesson 8.1 and measure them with the evaluation harness, so the grading calls you stubbed here become tracked, tuned quality gates.